In [ ]:
%run ./utils_common

In [ ]:
logger = setup_logger("AllPurposeDBUCostReporter")

In [ ]:
dbutils.widgets.text("catalog", "", "CATALOG")
dbutils.widgets.text("schema", "", "SCHEMA")
dbutils.widgets.text("overlap_days", "3", "Overlap days (min 2)")
dbutils.widgets.text("workspace_ids", "", "Workspace IDs (comma-separated, blank=all)")

In [ ]:
# =======================================================
# All-Purpose DBU Cost Client
# =======================================================
# Sibling of DBUCostClient (job clusters). Differences:
#   * cluster_source filter: IN ('UI','API')  (vs = 'JOB')
#   * usage filter:          job_run_id IS NULL  (vs IS NOT NULL)
#   * aggregation key:       (cluster_id, usage_date, workspace_id)
#                            -- no job_id / run_id columns
#   * user_id attribution:   COALESCE(cluster.owned_by, '__unknown__')
#   * surfaces data_security_mode so the UI can distinguish exact
#     (SINGLE_USER) from approximate (USER_ISOLATION) attribution
#   * MERGE key:             (cluster_id, user_id, usage_date)
class AllPurposeDBUCostClient:

    TABLE_NAME = "dbspend360_all_purpose_dbu_cost"

    def __init__(self, audit_table: str, target_table: str, overlap_days: int, logger=None):
        self.audit_table = audit_table
        self.target_table = target_table
        self.overlap_days = overlap_days
        self.logger = logger or logging.getLogger("AllPurposeDBUCostClient")
        raw_ws = dbutils.widgets.get("workspace_ids")
        if raw_ws.strip() == "":
            self.workspace_ids = None
        else:
            self.workspace_ids = [w.strip() for w in raw_ws.split(",") if w.strip()]

    def compute_and_merge_dbu_cost(self):
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

            valid, msg = validate_date_window(start_dt, end_dt)
            if not valid:
                raise DataQualityError(msg)

            self.logger.info(f"Loading all-purpose DBU cost from {start_dt} to {end_dt}")

            # SCD-collapse system.compute.clusters: pick the most-recent owned_by
            # and data_security_mode per cluster_id. max_by(col, change_time)
            # avoids dragging the full SCD history into the join.
            cluster_df = spark.sql("""
                SELECT cluster_id,
                       max_by(owned_by, change_time)            AS owned_by,
                       max_by(data_security_mode, change_time)  AS data_security_mode,
                       max_by(workspace_id, change_time)         AS cluster_workspace_id
                FROM system.compute.clusters
                WHERE cluster_source IN ('UI', 'API')
                GROUP BY cluster_id
            """)
            if self.workspace_ids is not None:
                cluster_df = cluster_df.filter(F.col("cluster_workspace_id").isin(self.workspace_ids))

            usage_df = (
                spark.table("system.billing.usage")
                     .alias("usage")
                     .filter(
                         (F.col("usage.usage_date") >= F.lit(start_dt)) &
                         (F.col("usage.usage_date") <= F.lit(end_dt))
                     )
            )
            if self.workspace_ids is not None:
                usage_df = usage_df.filter(F.col("usage.workspace_id").isin(self.workspace_ids))

            list_prices_df = spark.table("system.billing.list_prices").alias("list_prices")

            df = (
                usage_df.join(
                    list_prices_df,
                    on=(
                        (F.col("usage.sku_name") == F.col("list_prices.sku_name")) &
                        (F.col("usage.usage_start_time") >= F.col("list_prices.price_start_time")) &
                        (
                            (F.col("usage.usage_start_time") < F.col("list_prices.price_end_time")) |
                            F.col("list_prices.price_end_time").isNull()
                        )
                    ),
                    how="left"
                )
            )

            # Defense in depth: the cluster_source IN ('UI','API') filter on the
            # cluster_df join is the primary guard. The job_run_id IS NULL filter
            # is redundant for our scope (per the system.billing.usage reference,
            # job_run_id does not populate for jobs run on all-purpose compute)
            # but it keeps the source dataframe lean before the downstream join.
            filtered_df = df.filter(
                F.col("usage.usage_metadata")["job_run_id"].isNull()
            )

            agg_df = (
                filtered_df
                .groupBy(
                    F.col("usage.usage_metadata")["cluster_id"].alias("source_cluster_id"),
                    F.col("usage.usage_date").alias("usage_date"),
                    F.col("usage.workspace_id").alias("workspace_id")
                )
                .agg(
                    F.sum(
                        F.col("usage.usage_quantity")
                        * F.col("list_prices.pricing")["default"].cast("double")
                    ).alias("databricks_cost"),
                    F.concat_ws(
                        " + ",
                        F.array_sort(F.collect_set(F.col("usage.sku_name")))
                    ).alias("sku_name_merged")
                )
            )

            # Inner join filters usage rows down to only those whose cluster_id
            # has cluster_source IN ('UI','API'). Same shape as the job-cluster
            # pipeline's inner join against the JOB-source cluster set.
            joined_df = (
                agg_df.join(
                    cluster_df,
                    on=(agg_df["source_cluster_id"] == cluster_df["cluster_id"]),
                    how="inner"
                )
                .drop("source_cluster_id", "cluster_workspace_id")
            )

            joined_df = joined_df.withColumn("currency", F.lit("USD"))

            dbu_inc_df = (
                joined_df
                .select(
                    "cluster_id",
                    # Per plan §3.2: NULL owned_by (rare; SCD row exists but
                    # owner unresolvable) buckets to __unknown__ so the row
                    # remains visible and reconcilable.
                    F.coalesce(F.col("owned_by"), F.lit("__unknown__")).alias("user_id"),
                    "usage_date",
                    "databricks_cost",
                    "currency",
                    F.col("sku_name_merged").alias("sku_name"),
                    "workspace_id",
                    "data_security_mode",
                )
            )

            if dbu_inc_df.limit(1).count() == 0:
                self.logger.info(
                    "No all-purpose DBU rows after filtering / aggregation. "
                    "Verify there exist clusters with cluster_source IN ('UI','API') "
                    "having SKU-billed usage in the date window."
                )
                merged_row_count = 0
            else:
                dbu_inc_df = (
                    dbu_inc_df
                    .withColumn("created_at", F.current_timestamp())
                    .withColumn("updated_at", F.current_timestamp())
                )
                dbu_inc_df = safe_cache(dbu_inc_df)

                merged_row_count = dbu_inc_df.count()

                validate_source_schema(
                    dbu_inc_df,
                    {"cluster_id": "string", "user_id": "string",
                     "usage_date": "date", "databricks_cost": "double"},
                    self.target_table, self.logger,
                )
                validate_no_negative_costs(
                    dbu_inc_df, ["databricks_cost"], self.target_table, self.logger,
                )
                validate_currency_consistency(dbu_inc_df, "currency", self.target_table, self.logger)

                target = DeltaTable.forName(spark, self.target_table)
                (target.alias("t")
                    .merge(
                        dbu_inc_df.alias("s"),
                        "t.cluster_id = s.cluster_id AND t.user_id = s.user_id "
                        "AND t.usage_date = s.usage_date",
                    )
                    .whenMatchedUpdate(set={
                        "databricks_cost": "s.databricks_cost",
                        "data_security_mode": "s.data_security_mode",
                        "updated_at": "current_timestamp()",
                    })
                    .whenNotMatchedInsert(values={
                        "cluster_id": "s.cluster_id",
                        "user_id": "s.user_id",
                        "usage_date": "s.usage_date",
                        "databricks_cost": "s.databricks_cost",
                        "currency": "s.currency",
                        "sku_name": "s.sku_name",
                        "workspace_id": "s.workspace_id",
                        "data_security_mode": "s.data_security_mode",
                        "created_at": "current_timestamp()",
                        "updated_at": "current_timestamp()",
                    })
                    .execute()
                )

                safe_unpersist(dbu_inc_df)
                get_merge_metrics(self.target_table, self.logger)

                validate_post_merge(
                    self.target_table, "usage_date",
                    start_dt, end_dt, merged_row_count, self.logger,
                )

            log_audit_run(self.audit_table, self.TABLE_NAME, start_dt, end_dt, "SUCCESS", merged_row_count, "")
            self.logger.info(
                f"Merged {merged_row_count} rows into {self.target_table} "
                f"for {start_dt} \u2192 {end_dt}."
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Run failed: {msg}")
            try:
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt, "FAILED", 0, msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED audit entry")
            raise

In [ ]:
# =======================================================
# APP
# =======================================================
class AllPurposeDBUCostReporterApp:

    def __init__(self):
        catalog = dbutils.widgets.get("catalog")
        schema = dbutils.widgets.get("schema")
        overlap_days = get_overlap_days(dbutils.widgets.get("overlap_days"), logger=logger)

        audit_table = build_table_fqn(catalog, schema, "dbspend360_audit_log")
        target_table = build_table_fqn(catalog, schema, "dbspend360_all_purpose_dbu_cost")

        self.client = AllPurposeDBUCostClient(
            audit_table=audit_table,
            target_table=target_table,
            overlap_days=overlap_days,
            logger=logger,
        )

    def run(self):
        self.client.compute_and_merge_dbu_cost()

In [ ]:
# =======================================================
# Execute
# =======================================================
app = AllPurposeDBUCostReporterApp()
app.run()